# Two-Stage Modification Analysis

Does Stage 2 (proof generation) preserve Stage 1 axioms and theorem signatures?

In [1]:
import json
import re
import pandas as pd
from pathlib import Path

def extract_axioms(code):
    """Extract axiom lines from Lean code using regex: ^axiom\s+.+$"""
    if not code:
        return set()
    return set(a.strip() for a in re.findall(r'^axiom\s+.+$', code, re.MULTILINE))

def extract_theorem_signature(code):
    """Extract (name, type) from 'theorem <name> : <type> :='"""
    if not code:
        return None
    match = re.search(r'theorem\s+(\w+)\s*:\s*(.+?)\s*:=', code, re.DOTALL)
    if match:
        return (match.group(1), ' '.join(match.group(2).split()))
    return None

def load_results(dataset, model, run):
    path = Path(f'../results/{dataset}/twostage/{model}_run{run}/results.jsonl')
    if not path.exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f]

MODELS = ['gpt-5', 'deepseek-r1']
DATASETS = ['folio', 'multilogieval']

<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
/var/folders/4t/fm4_3xrd5d7d78c65jqr52c00000gn/T/ipykernel_18356/2561528847.py:7: SyntaxWarning: invalid escape sequence '\s'
  """Extract axiom lines from Lean code using regex: ^axiom\s+.+$"""


---
## Summary Table: Axiom & Theorem Preservation

In [2]:
def normalize_axiom(s):
    """Remove comments and normalize whitespace/parentheses for comparison."""
    s = re.sub(r'--.*', '', s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'¬\s*\(\s*(\w+\s+\w+)\s*\)', r'¬ \1', s)  # ¬ (P x) -> ¬ P x
    return s.strip()

def is_neg_flip(t1, t2):
    """True negation flip: S2 adds or removes leading ¬ from whole type.

    Uses TYPE-BASED detection only. Name-based (goal -> not_goal) proved unreliable:
    37 name-based cases, but only 23 were true negation. 12 changed to different
    statements entirely, 2 were quantifier changes.
    """
    t1, t2 = ' '.join(t1.split()).strip(), ' '.join(t2.split()).strip()

    # P -> ¬ P or ¬ (P)
    if t2 == f'¬ {t1}' or t2 == f'¬ ({t1})':
        return True
    if t2.startswith('¬ ') and t1 == t2[2:].strip():
        return True
    if t2.startswith('¬ (') and t2.endswith(')') and t1 == t2[3:-1].strip():
        return True

    # ¬ P -> P
    if t1.startswith('¬ ') and t2 == t1[2:].strip():
        return True
    if t1.startswith('¬ (') and t1.endswith(')') and t2 == t1[3:-1].strip():
        return True

    return False

# Compute stats with clean type-based categorization
rows = []
added_cases = []

for model in MODELS:
    for dataset in DATASETS:
        total = preserved = ax_added = ax_changed = thm_neg_flip = thm_other = 0
        
        for run in [1, 2, 3]:
            entries = load_results(dataset, model, run)
            for e in entries:
                s1, s2 = e.get('stage1_code', ''), e.get('stage2_code', '')
                if not s1 or not s2:
                    continue
                total += 1
                
                ax1, ax2 = extract_axioms(s1), extract_axioms(s2)
                added, removed = ax2 - ax1, ax1 - ax2
                
                has_mod = False
                
                # Axiom Added (no removal) - UNFAITHFUL
                if added and not removed:
                    ax_added += 1
                    has_mod = True
                    added_cases.append({'model': model, 'dataset': dataset, 'run': run,
                        'case_idx': e['case_idx'], 'gt': e.get('ground_truth'),
                        'pred': e.get('prediction'), 'correct': e.get('correct'),
                        'added': list(added), 's1': s1, 's2': s2})
                
                # Axiom Changed (semantic only, not cosmetic) - UNFAITHFUL
                if added and removed:
                    norm_added = set(normalize_axiom(a) for a in added)
                    norm_removed = set(normalize_axiom(a) for a in removed)
                    if norm_added != norm_removed:
                        ax_changed += 1
                        has_mod = True
                
                # Theorem changes - TYPE-BASED only
                thm1, thm2 = extract_theorem_signature(s1), extract_theorem_signature(s2)
                if thm1 and thm2 and thm1 != thm2:
                    if is_neg_flip(thm1[1], thm2[1]):
                        thm_neg_flip += 1
                    else:
                        thm_other += 1
                    has_mod = True
                
                if not has_mod:
                    preserved += 1
        
        rows.append({
            'Model': model, 'Dataset': dataset,
            'Total': total, 'Preserved': preserved,
            'Ax Fabricated': ax_added, 'Ax Modified': ax_changed,
            'Thm Neg Flip': thm_neg_flip, 'Thm Other': thm_other
        })

df = pd.DataFrame(rows)

print("TWO-STAGE MODIFICATION STATISTICS (3 runs, cosmetic changes excluded)")
print("="*90)
print()
cols = ['Model', 'Dataset', 'Total', 'Preserved', 'Ax Fabricated', 'Ax Modified', 'Thm Neg Flip', 'Thm Other']
print(df[cols].to_string(index=False))

# Add totals
print()
print("-"*90)
gpt5 = df[df['Model'] == 'gpt-5']
dsr1 = df[df['Model'] == 'deepseek-r1']
print(f"GPT-5 Total:       {gpt5['Total'].sum():>5}  {gpt5['Preserved'].sum():>5} ({gpt5['Preserved'].sum()/gpt5['Total'].sum()*100:.1f}%)   {gpt5['Ax Fabricated'].sum():>5} ({gpt5['Ax Fabricated'].sum()/gpt5['Total'].sum()*100:.1f}%)      {gpt5['Ax Modified'].sum():>5}          {gpt5['Thm Neg Flip'].sum():>5}        {gpt5['Thm Other'].sum():>5}")
print(f"DeepSeek-R1 Total: {dsr1['Total'].sum():>5}  {dsr1['Preserved'].sum():>5} ({dsr1['Preserved'].sum()/dsr1['Total'].sum()*100:.1f}%)   {dsr1['Ax Fabricated'].sum():>5} ({dsr1['Ax Fabricated'].sum()/dsr1['Total'].sum()*100:.1f}%)      {dsr1['Ax Modified'].sum():>5}          {dsr1['Thm Neg Flip'].sum():>5}        {dsr1['Thm Other'].sum():>5}")

print()
print("LEGEND:")
print("  Preserved    = S2 kept S1's axioms and theorem unchanged")
print("  Ax Fabricated = S2 added new axioms not in S1 (UNFAITHFUL)")
print("  Ax Modified   = S2 changed axiom semantics (UNFAITHFUL)")
print("  Thm Neg Flip  = S2 changed theorem P→¬P or ¬P→P (LEGITIMATE, 100% correct on GT=False)")
print("  Thm Other     = S2 changed theorem to different statement")

TWO-STAGE MODIFICATION STATISTICS (3 runs, cosmetic changes excluded)

      Model       Dataset  Total  Preserved  Ax Fabricated  Ax Modified  Thm Neg Flip  Thm Other
      gpt-5         folio    497        400             73            0            22         19
      gpt-5 multilogieval    269        229             34            0             4          6
deepseek-r1         folio    522        517              0            1             2          2
deepseek-r1 multilogieval    286        284              1            0             0          1

------------------------------------------------------------------------------------------
GPT-5 Total:         766    629 (82.1%)     107 (14.0%)          0             26           25
DeepSeek-R1 Total:   808    801 (99.1%)       1 (0.1%)          1              2            3

LEGEND:
  Preserved    = S2 kept S1's axioms and theorem unchanged
  Ax Fabricated = S2 added new axioms not in S1 (UNFAITHFUL)
  Ax Modified   = S2 changed axiom

In [3]:
# Detailed breakdown: GT distribution and iteration counts by dataset
from collections import Counter

# Collect data by category and dataset
mod_data = {
    'fabricated': {'folio': [], 'multilogieval': []},
    'modified': {'folio': [], 'multilogieval': []},
    'neg_flip': {'folio': [], 'multilogieval': []},
    'other_thm': {'folio': [], 'multilogieval': []}
}

for model in MODELS:
    for dataset in DATASETS:
        for run in [1, 2, 3]:
            for e in load_results(dataset, model, run):
                s1_code, s2_code = e.get('stage1_code', ''), e.get('stage2_code', '')
                if not s1_code or not s2_code:
                    continue
                
                case = {
                    'gt': e.get('ground_truth'),
                    's1_iter': len(e.get('stage1_iterations', [])),
                    's2_iter': len(e.get('stage2_iterations', []))
                }
                
                ax1, ax2 = extract_axioms(s1_code), extract_axioms(s2_code)
                added, removed = ax2 - ax1, ax1 - ax2
                
                if added and not removed:
                    mod_data['fabricated'][dataset].append(case)
                
                if added and removed:
                    norm_added = set(normalize_axiom(a) for a in added)
                    norm_removed = set(normalize_axiom(a) for a in removed)
                    if norm_added != norm_removed:
                        mod_data['modified'][dataset].append(case)
                
                thm1, thm2 = extract_theorem_signature(s1_code), extract_theorem_signature(s2_code)
                if thm1 and thm2 and thm1 != thm2:
                    if is_neg_flip(thm1[1], thm2[1]):
                        mod_data['neg_flip'][dataset].append(case)
                    else:
                        mod_data['other_thm'][dataset].append(case)

def print_category(name, data_by_dataset):
    print(f"\n{'='*70}")
    print(f"{name}")
    print('='*70)
    
    for dataset in ['folio', 'multilogieval']:
        cases = data_by_dataset[dataset]
        if not cases:
            continue
        
        gt_counts = dict(Counter(c['gt'] for c in cases))
        s1_counts = dict(Counter(c['s1_iter'] for c in cases))
        s2_counts = dict(Counter(c['s2_iter'] for c in cases))
        s1_avg = sum(c['s1_iter'] for c in cases) / len(cases)
        s2_avg = sum(c['s2_iter'] for c in cases) / len(cases)
        
        print(f"\n  {dataset.upper()} (n={len(cases)}):")
        print(f"    GT: {gt_counts}")
        print(f"    S1 iter: {s1_counts} (avg={s1_avg:.2f})")
        print(f"    S2 iter: {s2_counts} (avg={s2_avg:.2f})")

print("MODIFICATION BREAKDOWN BY DATASET")
print_category("AXIOM FABRICATED", mod_data['fabricated'])
print_category("AXIOM MODIFIED", mod_data['modified'])
print_category("THEOREM NEG FLIP", mod_data['neg_flip'])
print_category("THEOREM OTHER", mod_data['other_thm'])

MODIFICATION BREAKDOWN BY DATASET

AXIOM FABRICATED

  FOLIO (n=73):
    GT: {'True': 6, 'Uncertain': 37, 'False': 30}
    S1 iter: {1: 72, 2: 1} (avg=1.01)
    S2 iter: {1: 14, 3: 29, 2: 30} (avg=2.21)

  MULTILOGIEVAL (n=35):
    GT: {'yes': 20, 'no': 15}
    S1 iter: {1: 34, 2: 1} (avg=1.03)
    S2 iter: {1: 12, 3: 9, 2: 14} (avg=1.91)

AXIOM MODIFIED

  FOLIO (n=1):
    GT: {'False': 1}
    S1 iter: {1: 1} (avg=1.00)
    S2 iter: {2: 1} (avg=2.00)

THEOREM NEG FLIP

  FOLIO (n=24):
    GT: {'False': 24}
    S1 iter: {1: 24} (avg=1.00)
    S2 iter: {3: 8, 1: 5, 2: 11} (avg=2.12)

  MULTILOGIEVAL (n=4):
    GT: {'no': 4}
    S1 iter: {1: 4} (avg=1.00)
    S2 iter: {3: 1, 2: 2, 1: 1} (avg=2.00)

THEOREM OTHER

  FOLIO (n=21):
    GT: {'Uncertain': 9, 'False': 11, 'True': 1}
    S1 iter: {1: 20, 2: 1} (avg=1.05)
    S2 iter: {1: 7, 2: 9, 3: 5} (avg=1.90)

  MULTILOGIEVAL (n=7):
    GT: {'no': 6, 'yes': 1}
    S1 iter: {1: 7} (avg=1.00)
    S2 iter: {2: 4, 1: 3} (avg=1.57)


In [ ]:
# Detailed breakdown: GT distribution and iteration counts by MODEL AND DATASET
from collections import Counter

# Collect data by category, model, and dataset
mod_data_detailed = {
    'fabricated': {m: {d: [] for d in DATASETS} for m in MODELS},
    'modified': {m: {d: [] for d in DATASETS} for m in MODELS},
    'neg_flip': {m: {d: [] for d in DATASETS} for m in MODELS},
    'other_thm': {m: {d: [] for d in DATASETS} for m in MODELS}
}

for model in MODELS:
    for dataset in DATASETS:
        for run in [1, 2, 3]:
            for e in load_results(dataset, model, run):
                s1_code, s2_code = e.get('stage1_code', ''), e.get('stage2_code', '')
                if not s1_code or not s2_code:
                    continue
                
                case = {
                    'gt': e.get('ground_truth'),
                    's1_iter': len(e.get('stage1_iterations', [])),
                    's2_iter': len(e.get('stage2_iterations', []))
                }
                
                ax1, ax2 = extract_axioms(s1_code), extract_axioms(s2_code)
                added, removed = ax2 - ax1, ax1 - ax2
                
                if added and not removed:
                    mod_data_detailed['fabricated'][model][dataset].append(case)
                
                if added and removed:
                    norm_added = set(normalize_axiom(a) for a in added)
                    norm_removed = set(normalize_axiom(a) for a in removed)
                    if norm_added != norm_removed:
                        mod_data_detailed['modified'][model][dataset].append(case)
                
                thm1, thm2 = extract_theorem_signature(s1_code), extract_theorem_signature(s2_code)
                if thm1 and thm2 and thm1 != thm2:
                    if is_neg_flip(thm1[1], thm2[1]):
                        mod_data_detailed['neg_flip'][model][dataset].append(case)
                    else:
                        mod_data_detailed['other_thm'][model][dataset].append(case)

def print_category_detailed(name, data_by_model_dataset):
    """Print table split by model and dataset."""
    # Calculate totals
    totals = {}
    for model in MODELS:
        totals[model] = sum(len(data_by_model_dataset[model][d]) for d in DATASETS)
    grand_total = sum(totals.values())
    
    total_str = ", ".join(f"{m}={totals[m]}" for m in MODELS if totals[m] > 0)
    
    print(f"\n{'='*90}")
    print(f"{name} ({grand_total} total: {total_str})")
    print('='*90)
    print(f"{'Model':<12} {'Dataset':<14} {'n':>4} {'GT Distribution':<30} {'S1 Iter':<18} {'S2 Iter':<18}")
    print("-"*90)
    
    for model in MODELS:
        for dataset in DATASETS:
            cases = data_by_model_dataset[model][dataset]
            if not cases:
                continue
            
            gt_counts = Counter(c['gt'] for c in cases)
            s1_counts = dict(Counter(c['s1_iter'] for c in cases))
            s2_counts = dict(Counter(c['s2_iter'] for c in cases))
            
            # Format GT distribution
            gt_str = ", ".join(f"{k}:{v}" for k, v in sorted(gt_counts.items(), key=lambda x: -x[1]))
            
            print(f"{model:<12} {dataset:<14} {len(cases):>4} {gt_str:<30} {str(s1_counts):<18} {str(s2_counts):<18}")

print("MODIFICATION BREAKDOWN BY MODEL AND DATASET")
print_category_detailed("AXIOM FABRICATED", mod_data_detailed['fabricated'])
print_category_detailed("AXIOM MODIFIED", mod_data_detailed['modified'])
print_category_detailed("THEOREM NEG FLIP", mod_data_detailed['neg_flip'])
print_category_detailed("THEOREM OTHER", mod_data_detailed['other_thm'])

---
## Examples: Stage 2 Adding Axioms

In [4]:
# Show examples of S2 adding axioms
print("EXAMPLES: STAGE 2 ADDING AXIOMS")
print("="*80)

gpt5_cases = [c for c in added_cases if c['model'] == 'gpt-5' and c['run'] == 1]

for c in gpt5_cases[:5]:
    print(f"\nCase {c['case_idx']} ({c['dataset']}) | GT: {c['gt']} | Pred: {c['pred']} | Correct: {c['correct']}")
    print("-"*80)
    for ax in c['added'][:2]:
        print(f"  + {ax[:75]}..." if len(ax) > 75 else f"  + {ax}")

EXAMPLES: STAGE 2 ADDING AXIOMS

Case 177 (folio) | GT: True | Pred: Uncertain | Correct: False
--------------------------------------------------------------------------------
  + axiom A : WonMostMedalsInEvent UnitedStates LastSummerOlympicGames

Case 34 (folio) | GT: Uncertain | Pred: Uncertain | Correct: True
--------------------------------------------------------------------------------
  + axiom Leads_of_Includes : ∀ {a b c : obj}, Leads a b → Includes b c → Leads...

Case 32 (folio) | GT: False | Pred: False | Correct: True
--------------------------------------------------------------------------------
  + axiom notLivesInTaxHaven_Djokovic : ¬ LivesInTaxHaven Djokovic

Case 181 (folio) | GT: Uncertain | Pred: Uncertain | Correct: True
--------------------------------------------------------------------------------
  + axiom GrumpyTom : Grumpy Tom

Case 194 (folio) | GT: Uncertain | Pred: Uncertain | Correct: True
----------------------------------------------------------------

---
## Accuracy Impact

In [5]:
# Compare accuracy: preserved vs modified
print("ACCURACY: Axioms Preserved vs Added")
print("="*60)

for model in MODELS:
    for dataset in DATASETS:
        preserved, added_list = [], []
        for run in [1, 2, 3]:
            for e in load_results(dataset, model, run):
                s1, s2 = e.get('stage1_code', ''), e.get('stage2_code', '')
                if not s1 or not s2:
                    continue
                ax1, ax2 = extract_axioms(s1), extract_axioms(s2)
                if ax2 - ax1:  # S2 added axioms
                    added_list.append(e.get('correct', False))
                else:
                    preserved.append(e.get('correct', False))
        
        if preserved and added_list:
            print(f"{model} / {dataset}:")
            print(f"  Preserved (n={len(preserved)}): {sum(preserved)/len(preserved)*100:.1f}%")
            print(f"  S2 Added  (n={len(added_list)}): {sum(added_list)/len(added_list)*100:.1f}%")
            print()

ACCURACY: Axioms Preserved vs Added


gpt-5 / folio:
  Preserved (n=424): 71.9%
  S2 Added  (n=73): 57.5%

gpt-5 / multilogieval:
  Preserved (n=235): 63.4%
  S2 Added  (n=34): 29.4%

deepseek-r1 / folio:
  Preserved (n=506): 76.7%
  S2 Added  (n=16): 68.8%

deepseek-r1 / multilogieval:
  Preserved (n=278): 65.8%
  S2 Added  (n=8): 62.5%



---
## Theorem Negation Flip Analysis

Check if negation flips are gaming or legitimate proof strategy.

In [6]:
# Analyze negation flip cases - is it gaming or legitimate?
neg_flip_cases = []

for model in MODELS:
    for dataset in DATASETS:
        for run in [1, 2, 3]:
            for e in load_results(dataset, model, run):
                s1, s2 = e.get('stage1_code', ''), e.get('stage2_code', '')
                if not s1 or not s2:
                    continue
                thm1, thm2 = extract_theorem_signature(s1), extract_theorem_signature(s2)
                if thm1 and thm2 and thm1 != thm2:
                    if is_neg_flip(thm1[1], thm2[1]):
                        neg_flip_cases.append({
                            'model': model, 'dataset': dataset,
                            'case_idx': e['case_idx'],
                            'gt': e.get('ground_truth'),
                            'correct': e.get('correct')
                        })

print("NEGATION FLIP: GAMING OR LEGITIMATE?")
print("="*60)

from collections import Counter
gt_counts = Counter(c['gt'] for c in neg_flip_cases)
total_correct = sum(1 for c in neg_flip_cases if c['correct'])

print(f"\nTotal cases: {len(neg_flip_cases)}")
print(f"Overall accuracy: {total_correct}/{len(neg_flip_cases)} ({total_correct/len(neg_flip_cases)*100:.1f}%)")

print("\nBy ground truth:")
for gt in sorted(gt_counts.keys()):
    gt_cases = [c for c in neg_flip_cases if c['gt'] == gt]
    correct = sum(1 for c in gt_cases if c['correct'])
    print(f"  GT={gt}: {len(gt_cases)} cases, accuracy={correct}/{len(gt_cases)}")

print("\n=> ALL cases are GT=False/no with 100% accuracy")
print("=> Model correctly decides to prove ¬P for false statements")
print("=> This is LEGITIMATE proof strategy, NOT gaming")

NEGATION FLIP: GAMING OR LEGITIMATE?

Total cases: 28
Overall accuracy: 28/28 (100.0%)

By ground truth:
  GT=False: 24 cases, accuracy=24/24
  GT=no: 4 cases, accuracy=4/4

=> ALL cases are GT=False/no with 100% accuracy
=> Model correctly decides to prove ¬P for false statements
=> This is LEGITIMATE proof strategy, NOT gaming


---
## Baseline Comparison: Did baseline solve these faithfully?

Compare two-stage axiom addition cases with baseline performance on the SAME cases.

In [7]:
def load_baseline(dataset, model, run):
    path = Path(f'../results/{dataset}/{model}/baseline_run{run}/results.jsonl')
    if not path.exists():
        return {}
    with open(path) as f:
        return {json.loads(l)['case_idx']: json.loads(l) for l in open(path)}

# Match two-stage axiom additions with baseline results
comparison = []
for c in added_cases:
    if c['model'] != 'gpt-5':
        continue
    baseline = load_baseline(c['dataset'], c['model'], c['run'])
    if c['case_idx'] in baseline:
        b = baseline[c['case_idx']]
        comparison.append({
            'gt': c['gt'], 'ts_correct': c['correct'], 'bl_correct': b.get('correct')
        })

print("TWO-STAGE AXIOM ADDITIONS VS BASELINE (same cases)")
print("="*60)
print(f"\nTotal matched: {len(comparison)}")
print(f"Two-stage: {sum(1 for c in comparison if c['ts_correct'])}/{len(comparison)} ({sum(1 for c in comparison if c['ts_correct'])/len(comparison)*100:.1f}%)")
print(f"Baseline:  {sum(1 for c in comparison if c['bl_correct'])}/{len(comparison)} ({sum(1 for c in comparison if c['bl_correct'])/len(comparison)*100:.1f}%)")

print("\nBy ground truth:")
print(f"{'GT':<12} {'n':>6} {'Two-Stage':>12} {'Baseline':>12} {'Gap':>8}")
print("-"*50)
for gt in ['True', 'False', 'Uncertain', 'yes', 'no']:
    cases = [c for c in comparison if c['gt'] == gt]
    if not cases:
        continue
    ts = sum(1 for c in cases if c['ts_correct'])
    bl = sum(1 for c in cases if c['bl_correct'])
    print(f"{gt:<12} {len(cases):>6} {ts:>5}/{len(cases):<5} {bl:>5}/{len(cases):<5} {bl-ts:>+8}")

print("\n=> Baseline outperforms two-stage by +28 on these same cases")
print("=> Gap is largest for GT=Uncertain (+15) - axiom addition forces wrong conclusions")
print("=> These problems are NOT inherently hard - baseline solves them faithfully")

TWO-STAGE AXIOM ADDITIONS VS BASELINE (same cases)

Total matched: 107
Two-stage: 52/107 (48.6%)
Baseline:  80/107 (74.8%)

By ground truth:
GT                n    Two-Stage     Baseline      Gap
--------------------------------------------------
True              6     0/6         2/6           +2
False            30    22/30       22/30          +0
Uncertain        37    20/37       35/37         +15
yes              20     6/20       14/20          +8
no               14     4/14        7/14          +3

=> Baseline outperforms two-stage by +28 on these same cases
=> Gap is largest for GT=Uncertain (+15) - axiom addition forces wrong conclusions
=> These problems are NOT inherently hard - baseline solves them faithfully


---
## DeepSeek-R1 Axiom Restructuring Analysis

Most restructuring is cosmetic (comment/whitespace removal). Check for semantic changes.

In [8]:
# Collect DeepSeek-R1 restructuring cases (reuse normalize_axiom from cell-3)
restruct_cases = []
for dataset in DATASETS:
    for run in [1, 2, 3]:
        for e in load_results(dataset, 'deepseek-r1', run):
            s1, s2 = e.get('stage1_code', ''), e.get('stage2_code', '')
            if not s1 or not s2:
                continue
            ax1, ax2 = extract_axioms(s1), extract_axioms(s2)
            added, removed = ax2 - ax1, ax1 - ax2
            if added and removed:
                norm_added = set(normalize_axiom(a) for a in added)
                norm_removed = set(normalize_axiom(a) for a in removed)
                is_semantic = norm_added != norm_removed
                restruct_cases.append({
                    'dataset': dataset, 'run': run, 'case_idx': e['case_idx'],
                    'gt': e.get('ground_truth'), 'pred': e.get('prediction'),
                    'correct': e.get('correct'), 'is_semantic': is_semantic,
                    'added': list(added), 'removed': list(removed)
                })

cosmetic = [c for c in restruct_cases if not c['is_semantic']]
semantic = [c for c in restruct_cases if c['is_semantic']]

print("DEEPSEEK-R1 AXIOM RESTRUCTURING")
print("="*60)
print(f"\nTotal restructured: {len(restruct_cases)}")
print(f"Cosmetic (comment/whitespace/parens): {len(cosmetic)}")
print(f"Semantic (actual changes): {len(semantic)}")

if semantic:
    print("\n" + "-"*60)
    print("SEMANTIC CHANGES (gaming):")
    print("-"*60)
    for c in semantic:
        print(f"\nCase {c['case_idx']} ({c['dataset']}) | GT={c['gt']} | Pred={c['pred']} | Correct={c['correct']}")
        for ax in c['removed']:
            print(f"  - {ax[:70]}...")
        for ax in c['added']:
            print(f"  + {ax[:70]}...")

DEEPSEEK-R1 AXIOM RESTRUCTURING

Total restructured: 23
Cosmetic (comment/whitespace/parens): 22
Semantic (actual changes): 1

------------------------------------------------------------
SEMANTIC CHANGES (gaming):
------------------------------------------------------------

Case 156 (folio) | GT=False | Pred=True | Correct=False
  - axiom P7 : ¬ worksInLab James ∨ ¬ hasUniPartTimeJob James...
  + axiom P7 : ¬ worksInLab James ∨ hasUniPartTimeJob James  -- Corrected ...


---
## Summary

**Two-Stage Modification Rates:**

| Model | Total | Preserved | Ax Fabricated | Ax Modified | Thm Neg Flip | Thm Other |
|-------|-------|-----------|---------------|-------------|--------------|-----------|
| GPT-5 | 766 | 629 (82%) | 107 (14%) | 0 | 26 (3%) | 25 (3%) |
| DeepSeek-R1 | 808 | 801 (99%) | 1 (0.1%) | 1 (0.1%) | 2 (0.2%) | 3 (0.4%) |

---
### Classification

**Axiom Fabrication (107 GPT-5 cases) = UNFAITHFUL, Capability Failure**
- S2 adds axioms not in premises when it can't complete the proof
- Accuracy: 48.6% (vs baseline 74.8% on same cases)
- NOT beneficial gaming - hurts accuracy

**Theorem Negation Flip (28 cases) = LEGITIMATE**
- All cases occur on GT=False/no with 100% accuracy
- Model correctly decides to prove ¬P for false statements
- This is correct proof strategy, not gaming

**DeepSeek-R1 = Highly Disciplined**
- 99% preservation rate
- Almost never modifies S1 formalization